In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/comment-category-prediction-challenge/Sample.csv
/kaggle/input/competitions/comment-category-prediction-challenge/train.csv
/kaggle/input/competitions/comment-category-prediction-challenge/test.csv


# 3. FEATURE ENGINEERING

In [10]:
def text_features(df):
    df["comment"] = df["comment"].fillna("")
    c = df["comment"]
    df["comment_len"] = c.str.len()
    df["word_count"] = c.str.split().apply(len)
    df["avg_word_len"] = df["comment_len"] / (df["word_count"] + 1)
    df["unique_word_ratio"] = c.apply(lambda s: len(set(s.split())) / (len(s.split()) + 1))
    df["upper_ratio"] = c.apply(lambda s: sum(ch.isupper() for ch in s) / (len(s) + 1))
    df["digit_ratio"] = c.apply(lambda s: sum(ch.isdigit() for ch in s) / (len(s) + 1))
    df["excl_count"] = c.str.count("!")
    df["quest_count"] = c.str.count(r"\?")
    df["punct_count"] = c.apply(lambda s: sum(ch in ".,;:!?" for ch in s))
    return df

def date_features(df):
    df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
    df["hour"] = df["created_date"].dt.hour
    df["dayofweek"] = df["created_date"].dt.dayofweek
    df["month"] = df["created_date"].dt.month
    for col in ["hour", "dayofweek", "month"]:
        df[col] = df[col].fillna(-1)
    return df

def engineer(df):
    df = df.copy()
    df = text_features(df)
    df = date_features(df)
    up = pd.to_numeric(df.get("upvote", 0), errors="coerce").fillna(0)
    down = pd.to_numeric(df.get("downvote", 0), errors="coerce").fillna(0)
    df["total_votes"] = up + down
    df["vote_ratio"] = (up - down) / (df["total_votes"] + 1)
    df["log_total_votes"] = np.log1p(df["total_votes"])
    return df

train = engineer(train)
test = engineer(test)

if "post_id" in train.columns:
    combined_counts = pd.concat([train["post_id"], test["post_id"]]).value_counts()
    train["post_comment_count"] = train["post_id"].map(combined_counts)
    test["post_comment_count"] = test["post_id"].map(combined_counts)

num_cols = [
    "emoticon_1", "emoticon_2", "emoticon_3", "upvote", "downvote", "if_1", "if_2",
    "race", "religion", "gender", "disability",
    "comment_len", "word_count", "avg_word_len", "unique_word_ratio",
    "upper_ratio", "digit_ratio", "excl_count", "quest_count", "punct_count",
    "hour", "dayofweek", "month", "total_votes", "vote_ratio", "log_total_votes",
    "post_comment_count",
]
num_cols = [c for c in num_cols if c in train.columns]
for c in num_cols:
    train[c] = pd.to_numeric(train[c], errors="coerce").fillna(0)
    test[c] = pd.to_numeric(test[c], errors="coerce").fillna(0)

le = LabelEncoder()
y = le.fit_transform(train["label"])
n_classes = len(le.classes_)
print("classes:", list(le.classes_))
print("class balance (train):")
print(pd.Series(y).value_counts(normalize=True).sort_index())


classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
class balance (train):
0    0.576631
1    0.080394
2    0.315354
3    0.027621
Name: proportion, dtype: float64


# 4. TEXT VECTORIZATION (TF-IDF, word-level)

In [11]:
word_tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=3,
                              sublinear_tf=True, stop_words="english")
Xw_train = word_tfidf.fit_transform(train["comment"])
Xw_test = word_tfidf.transform(test["comment"])
print("TF-IDF done:", elapsed(), "s elapsed —", Xw_train.shape[1], "features")


TF-IDF done: 48.7 s elapsed — 10000 features
